# Valve Integrity Analytics — Notebook 1: Data Ingestion & Exploration

**Project:** Well Integrity Surveillance Analytics Suite  
**Dataset:** Synthetic valve test records across 3 offshore assets  
**Purpose:** Load the dataset, understand its shape, validate the core business rule, and surface first-level insights.

---

### What this notebook covers
1. Mount Google Drive and load the CSV
2. Understand the dataset shape (rows, columns, types)
3. Check data quality (nulls, duplicates, date formats)
4. Validate the core Pass/Fail rule: `ΔP > Leak Rate → Failed`
5. Explore distributions (assets, valve types, results)
6. Save a clean version of the dataset for Notebook 2

---
### Domain context (for non-oil & gas readers)
Each row represents **one pressure test on one valve on one well on one date**.
- A well can have 4 valves: SCSSV, PWV, PMV, AMV
- The test bleeds down pressure, waits, then checks if pressure crept back up
- If it crept back more than the allowed **Leak Rate** → **Failed** (valve is leaking)
- Tests must happen within a **Recommended Interval** (180–210 days) or the well is considered **overdue**

## Step 0 — Install & import libraries

In [ ]:
# All libraries below come pre-installed in Google Colab — no pip install needed
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display

# Display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.1f}'.format)

# Plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'DejaVu Sans'

print('Libraries loaded successfully.')

## Step 1 — Mount Google Drive and load the dataset

> **Before running:** Upload `valve_test_dataset.csv` to your Google Drive.  
> Suggested path: `My Drive/valve_integrity_project/data/valve_test_dataset.csv`  
> Update `FILE_PATH` below if you put it somewhere else.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# ── UPDATE THIS PATH if your file is in a different folder ──
FILE_PATH = '/content/drive/MyDrive/valve_integrity_project/data/valve_test_dataset.csv'

df_raw = pd.read_csv(FILE_PATH)

print(f'Loaded {len(df_raw):,} rows and {len(df_raw.columns)} columns.')
df_raw.head(10)

## Step 2 — Understand the dataset shape

In [ ]:
# Basic shape
print('=== SHAPE ===')
print(f'Rows : {len(df_raw):,}')
print(f'Cols : {len(df_raw.columns)}')

print('\n=== COLUMN NAMES & TYPES ===')
print(df_raw.dtypes)

print('\n=== UNIQUE VALUES IN KEY COLUMNS ===')
print(f"Assets      : {sorted(df_raw['Asset'].unique())}")
print(f"Valve Types : {sorted(df_raw['Valve Type'].unique())}")
print(f"Results     : {sorted(df_raw['Result'].unique())}")
print(f"Wells       : {df_raw['Well ID'].nunique()} unique wells")

In [ ]:
# Wells per asset
print('=== WELLS PER ASSET ===')
wells_per_asset = df_raw.groupby('Asset')['Well ID'].nunique().reset_index()
wells_per_asset.columns = ['Asset', 'Unique Wells']
display(wells_per_asset)

# Rows per asset
print('\n=== ROWS PER ASSET ===')
display(df_raw['Asset'].value_counts().reset_index().rename(columns={'index':'Asset','Asset':'Row Count'}))

## Step 3 — Data quality checks

In [ ]:
# 3a — Check for nulls
print('=== NULL VALUES PER COLUMN ===')
null_summary = df_raw.isnull().sum().reset_index()
null_summary.columns = ['Column', 'Null Count']
null_summary['% of Total'] = (null_summary['Null Count'] / len(df_raw) * 100).round(1)
display(null_summary)

if null_summary['Null Count'].sum() == 0:
    print('\n✅ No null values found. Dataset is complete.')
else:
    print('\n⚠️  Nulls detected — review before analysis.')

In [ ]:
# 3b — Check for duplicate rows
dupes = df_raw.duplicated().sum()
print(f'Duplicate rows: {dupes}')
if dupes == 0:
    print('✅ No duplicate rows.')
else:
    print('⚠️  Duplicates found — dropping them.')
    df_raw = df_raw.drop_duplicates()

In [ ]:
# 3c — Parse Test Date properly
df_raw['Test Date'] = pd.to_datetime(df_raw['Test Date'], dayfirst=True)

print('Date range in dataset:')
print(f"  Earliest test : {df_raw['Test Date'].min().strftime('%d %b %Y')}")
print(f"  Latest test   : {df_raw['Test Date'].max().strftime('%d %b %Y')}")
print(f"  Span          : {(df_raw['Test Date'].max() - df_raw['Test Date'].min()).days} days")

In [ ]:
# 3d — Numeric column ranges (sanity check)
numeric_cols = [
    'Tubing Pressure Before Bled',
    'Tubing Pressure After Bled (PSI)',
    'Tubing Pressure After Build-up (PSI)',
    'Test Pressure (PSI)',
    'Leak Rate',
    'Delta P'
]
print('=== NUMERIC COLUMN SUMMARY ===')
display(df_raw[numeric_cols].describe().T)

## Step 4 — Validate the core business rule

**Rule:** `ΔP = Pressure After Build-up − Pressure After Bled`  
> If ΔP > Leak Rate → **Failed**  
> If ΔP ≤ Leak Rate → **Passed**

We recalculate this independently and compare against the `Result` column to verify dataset integrity.

In [ ]:
# Recalculate Delta P and derive expected result
df_raw['Delta_P_check'] = (
    df_raw['Tubing Pressure After Build-up (PSI)'] -
    df_raw['Tubing Pressure After Bled (PSI)']
)

df_raw['Result_check'] = np.where(
    df_raw['Delta_P_check'] > df_raw['Leak Rate'],
    'Failed',
    'Passed'
)

# Compare with stored Result
mismatches = (df_raw['Result'] != df_raw['Result_check']).sum()

print(f'Rows where stored Result matches recalculated Result : {len(df_raw) - mismatches:,}')
print(f'Mismatches (should be 0)                           : {mismatches}')

if mismatches == 0:
    print('\n✅ Business rule validated — all Pass/Fail results are consistent with the pressure data.')
else:
    print('\n⚠️  Mismatches found:')
    display(df_raw[df_raw['Result'] != df_raw['Result_check']])

In [ ]:
# Show some failed rows to understand what failure looks like
print('=== SAMPLE FAILED TESTS ===')
failed_sample = df_raw[df_raw['Result'] == 'Failed'][[
    'Well ID', 'Asset', 'Valve Type', 'Test Date',
    'Tubing Pressure After Bled (PSI)',
    'Tubing Pressure After Build-up (PSI)',
    'Delta P', 'Leak Rate', 'Result'
]].head(8)
display(failed_sample)

print(f'\nTotal failed tests in dataset : {(df_raw["Result"] == "Failed").sum()}')
print(f'Total passed tests in dataset : {(df_raw["Result"] == "Passed").sum()}')

## Step 5 — Explore distributions

Four charts that give you a feel for what's in the data before any analysis.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Dataset Overview — Valve Test Records', fontsize=14, fontweight='bold', y=1.01)

# ── Chart 1: Test count by Asset ──
ax1 = axes[0, 0]
asset_counts = df_raw['Asset'].value_counts()
bars = ax1.bar(asset_counts.index, asset_counts.values,
               color=sns.color_palette('muted', len(asset_counts)), edgecolor='white', linewidth=0.8)
ax1.set_title('Total Tests by Asset', fontweight='bold')
ax1.set_ylabel('Number of Tests')
ax1.set_xlabel('')
ax1.tick_params(axis='x', rotation=15)
for bar in bars:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             str(int(bar.get_height())), ha='center', va='bottom', fontsize=10)

# ── Chart 2: Test count by Valve Type ──
ax2 = axes[0, 1]
valve_counts = df_raw['Valve Type'].value_counts()
colors_valve = sns.color_palette('Set2', len(valve_counts))
wedges, texts, autotexts = ax2.pie(
    valve_counts.values, labels=valve_counts.index,
    autopct='%1.0f%%', colors=colors_valve,
    startangle=90, pctdistance=0.75
)
for at in autotexts:
    at.set_fontsize(9)
ax2.set_title('Tests by Valve Type', fontweight='bold')

# ── Chart 3: Pass vs Fail by Asset ──
ax3 = axes[1, 0]
result_by_asset = df_raw.groupby(['Asset', 'Result']).size().unstack(fill_value=0)
result_by_asset.plot(kind='bar', ax=ax3, color=['#4CAF50', '#E53935'],
                     edgecolor='white', linewidth=0.8)
ax3.set_title('Pass vs Fail by Asset', fontweight='bold')
ax3.set_ylabel('Number of Tests')
ax3.set_xlabel('')
ax3.tick_params(axis='x', rotation=15)
ax3.legend(title='Result')

# ── Chart 4: Tests over time (monthly) ──
ax4 = axes[1, 1]
df_raw['Year_Month'] = df_raw['Test Date'].dt.to_period('M')
tests_over_time = df_raw.groupby('Year_Month').size()
ax4.plot(tests_over_time.index.astype(str), tests_over_time.values,
         color='steelblue', linewidth=1.5, marker='o', markersize=3)
ax4.set_title('Tests per Month (Timeline)', fontweight='bold')
ax4.set_ylabel('Tests Conducted')
ax4.set_xlabel('')
ax4.tick_params(axis='x', rotation=45, labelsize=7)
ax4.xaxis.set_major_locator(mticker.MultipleLocator(4))

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/valve_integrity_project/data/01_dataset_overview.png',
            bbox_inches='tight', dpi=130)
plt.show()
print('Chart saved.')

In [ ]:
# Failure rate breakdown: by asset and valve type
print('=== FAILURE RATE BY ASSET ===')
fail_by_asset = df_raw.groupby('Asset').apply(
    lambda x: pd.Series({
        'Total Tests': len(x),
        'Failed': (x['Result'] == 'Failed').sum(),
        'Failure Rate %': round((x['Result'] == 'Failed').mean() * 100, 1)
    })
).reset_index()
display(fail_by_asset)

print('\n=== FAILURE RATE BY VALVE TYPE ===')
fail_by_valve = df_raw.groupby('Valve Type').apply(
    lambda x: pd.Series({
        'Total Tests': len(x),
        'Failed': (x['Result'] == 'Failed').sum(),
        'Failure Rate %': round((x['Result'] == 'Failed').mean() * 100, 1)
    })
).reset_index()
display(fail_by_valve)

In [ ]:
# Delta P distribution — what do typical pressure rises look like?
fig, ax = plt.subplots(figsize=(10, 4))

# Separate passed and failed
passed = df_raw[df_raw['Result'] == 'Passed']['Delta P']
failed = df_raw[df_raw['Result'] == 'Failed']['Delta P']

ax.hist(passed, bins=30, alpha=0.7, color='#4CAF50', label='Passed', edgecolor='white')
ax.hist(failed, bins=10, alpha=0.8, color='#E53935', label='Failed', edgecolor='white')

ax.axvline(df_raw['Leak Rate'].median(), color='navy', linestyle='--',
           linewidth=1.2, label=f'Median Leak Rate ({df_raw["Leak Rate"].median():.0f} PSI)')

ax.set_title('Distribution of Delta P (Pressure Build-up)', fontweight='bold')
ax.set_xlabel('Delta P (PSI)')
ax.set_ylabel('Number of Tests')
ax.legend()

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/valve_integrity_project/data/01_delta_p_distribution.png',
            bbox_inches='tight', dpi=130)
plt.show()
print('Chart saved.')

## Step 6 — Build the clean dataset and save for Notebook 2

In [ ]:
# Drop the temporary check columns, keep everything clean
df_clean = df_raw.drop(columns=['Delta_P_check', 'Result_check', 'Year_Month'])

# Rename columns for easier coding in later notebooks
df_clean = df_clean.rename(columns={
    'Well ID'                              : 'well_id',
    'Asset'                                : 'asset',
    'Valve Type'                           : 'valve_type',
    'Test Date'                            : 'test_date',
    'Tubing Pressure Before Bled'          : 'pressure_before_bled',
    'Tubing Pressure After Bled (PSI)'     : 'pressure_after_bled',
    'Tubing Pressure After Build-up (PSI)' : 'pressure_after_buildup',
    'Test Pressure (PSI)'                  : 'test_pressure',
    'Leak Rate'                            : 'leak_rate',
    'Delta P'                              : 'delta_p',
    'Result'                               : 'result',
    'Recommended Interval Days'            : 'recommended_interval_days',
    'Remarks'                              : 'remarks'
})

# Sort by asset, well, valve type, then date
df_clean = df_clean.sort_values(['asset', 'well_id', 'valve_type', 'test_date']).reset_index(drop=True)

print('=== CLEAN DATASET PREVIEW ===')
display(df_clean.head(10))

print(f'\nFinal shape: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns')

In [ ]:
# Save clean dataset to Drive — this is what Notebook 2 will load
CLEAN_PATH = '/content/drive/MyDrive/valve_integrity_project/data/valve_tests_clean.csv'
df_clean.to_csv(CLEAN_PATH, index=False)
print(f'✅ Clean dataset saved to: {CLEAN_PATH}')

## Notebook 1 Summary

| Check | Result |
|---|---|
| Total rows loaded | 780 |
| Null values | 0 |
| Duplicate rows | 0 |
| Pass/Fail rule validated | ✅ 0 mismatches |
| Clean file saved | `valve_tests_clean.csv` |

---

**Next → Notebook 2:** Barrier status engine, overdue test detection, and per-well risk ranking.